# Cats vs Dogs Image Classification — Deep Transfer Learning (MobileNetV2)
**Repository:** `AIML_1_ImageClassification_byte`  
**Internship Track:** AVIP 2026 — AI/ML (Task 1)  
**Hardware Target:** Free Google Colab T4 GPU / Local CPU  

This notebook demonstrates deep transfer learning using a pre-trained **MobileNetV2** backbone trained on ImageNet, fine-tuned on the 25,000-image Kaggle Dogs vs. Cats competition dataset.

### Step 1: Install & Import Deep Learning Libraries
Uncomment the pip command below if running in a fresh Colab environment.

In [ ]:
# !pip install tensorflow matplotlib seaborn
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

### Step 2: Download Dataset in Colab
Using TensorFlow Datasets (TFDS) to automatically download and prepare the Dogs vs. Cats dataset with a single line of code.

In [ ]:
import tensorflow_datasets as tfds

(raw_train, raw_val, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)
print(f"Train samples: {len(raw_train)}, Val samples: {len(raw_val)}, Test samples: {len(raw_test)}")

### Step 3: Image Preprocessing & Data Augmentation
We resize all inputs to 160x160 and apply random horizontal flipping and rotation to mitigate overfitting.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def format_example(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

train_ds = raw_train.map(format_example).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = raw_val.map(format_example).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = raw_test.map(format_example).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

### Step 4: Transfer Learning Model Architecture
We load MobileNetV2 with pre-trained ImageNet weights, freeze the convolutional base, and attach a global average pooling layer and classification head.

In [ ]:
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

### Step 5: Model Training
Train for 10 epochs with early stopping on validation loss.

In [ ]:
history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc*100:.2f}%")